# Metinden hedef splat · SDXL → TRELLIS

Tek bir nesne tarifinden Scene Editor / Spark için standart **3DGS target.ply** üretir.
Örneğin `a red sports car`. **L4 24 GB veya A100** seçin, ardından **Çalışma zamanı → Tümünü çalıştır**.
T4 / yerel 8 GB GPU desteklenmez. Model ve ortam indirmeleri birkaç GB; yerel Colab diskinde en az 35 GiB gerekir.

[Microsoft TRELLIS](https://github.com/microsoft/TRELLIS) görüntü koşullu modeli ayrıntı için önerir.
[SDXL](https://huggingface.co/stabilityai/stable-diffusion-xl-base-1.0) referans görüntüyü üretir.
TRELLIS MIT; SDXL CreativeML Open RAIL++-M koşulları geçerlidir (kullanım kısıtlamalarına uyun).
Bu tarif yerelde çevrimdışı test edilmiştir; temiz Colab GPU üzerinde uçtan uca doğrulama henüz yapılmadı.


## 1. Ayarlar

İngilizce, tek nesneli kısa istem önerilir. NEGATIVE_PROMPT istenmeyen ayrıntıları, STYLE görünümü belirtir.
SEED tekrar denemeleri karşılaştırır; farklı GPU'larda bit düzeyinde aynı çıktı garanti edilmez.
baseline: L4, 25/12/12 adım · quality: L4, 40/20/20 · ultra: A100, 50/25/25 (SDXL/seyrek yapı/latent).
Ultra daha iyi sonuç garantisi değildir. TARGET_SPLAT_COUNT isteğe bağlı **üst sınırdır**: eksik splat üretilmez,
fazlası deterministik ve kayıplı seçilir; model belleğini azaltmaz. None orijinal sayıyı korur.
OUTPUT_DIR, MyDrive altında tam bir klasör yoludur. Aynı ayarlar ve dosya sağlama toplamlarıyla devam edilir.
İstemi/seed/ön ayarı değiştirirken yeni çıktı klasörü seçin; eski sonuçların üzerine yazılmaz.


In [ ]:
PROMPT = 'a red sports car'
NEGATIVE_PROMPT = ''
STYLE = ''
SEED = 42
QUALITY_PRESET = 'baseline'
OUTPUT_DIR = '/content/drive/MyDrive/GaussianTests/text_to_splat/red-sports-car'
TARGET_SPLAT_COUNT = None


## 2. Taşınabilir tarif ve GPU ön kontrolü

Tarif kaynakları notebook içine gömülüdür; uygulamanın GitHub dalını indirmeniz gerekmez.
Colab'ın Python 3.13 çekirdeği korunur; model adımları özel Python 3.11.11 ortamında çalışır.


In [ ]:
RUNTIME_FILES = {}
QUALITY_PRESETS = {}

import hashlib
import importlib
import json
import sys
from pathlib import Path

WORK = Path('/content/text-to-splat-v1')
if not RUNTIME_FILES or QUALITY_PRESET not in QUALITY_PRESETS:
    raise RuntimeError('Notebook Generator üzerinden notebook oluşturun; boş şablon doğrudan çalıştırılamaz.')
WORK.mkdir(parents=True, exist_ok=True)
for name, source in RUNTIME_FILES.items():
    if Path(name).name != name or not name.startswith('text_to_splat_'):
        raise RuntimeError('Gömülü tarif dosya adı geçersiz.')
    if name.endswith('.py'):
        compile(source, name, 'exec')
    (WORK / name).write_text(source, encoding='utf8')
if str(WORK) not in sys.path:
    sys.path.insert(0, str(WORK))
import text_to_splat_bootstrap
bootstrap = importlib.reload(text_to_splat_bootstrap)
CONFIG = {
    'prompt': PROMPT, 'negative_prompt': NEGATIVE_PROMPT, 'style': STYLE,
    'seed': SEED, 'quality_preset': QUALITY_PRESET, 'quality': QUALITY_PRESETS[QUALITY_PRESET],
    'output_dir': OUTPUT_DIR, 'target_splat_count': TARGET_SPLAT_COUNT, 'work_dir': str(WORK),
    'recipe_sha256': hashlib.sha256(json.dumps(RUNTIME_FILES, sort_keys=True).encode()).hexdigest(),
}
bootstrap.host_preflight(CONFIG)


## 3. Google Drive bağlantısı

Colab'ın Drive yetkilendirme penceresini tamamlayın. Ara görüntü, Gaussian verisi, PLY, GIF,
manifest ve günlükler OUTPUT_DIR altında tutulur. Ayarlar uyuşmazsa yeni klasör seçmeniz istenir.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
bootstrap.validate_config(CONFIG)
print('Çıktı klasörü:', OUTPUT_DIR)


## 4. Sabitlenmiş ortam · yalnız hazır wheel paketleri

Python 3.11.11 · Torch 2.4.0 cu121 · xformers 0.0.27.post2 · spconv-cu120 2.3.6.
Kaolin/nvdiffrast/flash-attn veya mesh rasterizer kurulmaz. Kernel yeniden başlatılmaz.
Paket wheel bulunamazsa derlemek yerine hata verilir. Minik CUDA kontrolleri büyük model indirmelerinden önce çalışır.
İlk kurulum indirme hızına bağlıdır; aynı Colab oturumunda sonraki çalıştırma ortamı kullanır.


In [ ]:
bootstrap.prepare_environment(CONFIG)

## 5. Metin → referans görüntü

SDXL FP16 görüntü üretir; CPU arka plan temizliği input.png ve cutout.png dosyalarını kaydeder.
Bu adım bittiğinde GPU işlemi kapanır; TRELLIS ile aynı anda model tutmaz.
Görüntüde nesne eksikse/yanlışsa istemi değiştirip yeni OUTPUT_DIR kullanın.


In [ ]:
bootstrap.run_stage(CONFIG, 'image')
from IPython.display import display, Image
display(Image(filename=str(Path(OUTPUT_DIR) / 'input.png'), width=384))

## 6. Görüntü → gerçek 3D Gaussian

Yalnız TRELLIS Gaussian decoder ağırlıkları yüklenir. Mesh veya COLMAP eğitimi yapılmaz.
Tamamlanan aşamalar manifest + SHA256 ile doğrulanır; kesilen aşama yeniden başlar.
Colab oturumu yenilenirse ortam ve model önbelleği yeniden indirilir, Drive'daki tamamlanan aşamalar korunur.


In [ ]:
bootstrap.run_stage(CONFIG, 'gaussian')

## 7. Standart PLY ve döner önizleme

Konumlar orijine merkezlenir; en uzun Gaussian merkez sınırı 1 birimdir. Sağ elli **+Y yukarı**:
TRELLIS (x,y,z) → (x,z,-y). Quaternionlar aynı rotasyonla dönüştürülür.
PLY: opacity logit, scale log, rot wxyz; gerçek SH0 renk, 45 sıfır f_rest alanı.
COLMAP dünya ekseni ve ölçeği keyfidir; bu nesne metre cinsinden değildir. Scene Editor'de döndürün/ölçekleyin.
GIF CPU'da en önemli 12.000 Gaussian ile yaklaşık ortografik SH0 önizlemedir; tam PLY daha ayrıntılı olabilir.


In [ ]:
bootstrap.run_stage(CONFIG, 'export')
display(Image(filename=str(Path(OUTPUT_DIR) / 'turntable.gif')))

## 8. Sonuçları uygulamada kullanın

Drive → OUTPUT_DIR → **target.ply** dosyasını bilgisayarınıza indirin.
Scene Editor'de PLY'yi varlık olarak içe aktarın ve sahneye ekleyin; orijindeki nesneyi mevcut sahnenize göre hizalayın.
Morph panelinde kaynak A olarak myroom, hedef B olarak bu varlığı seçin. Önce küçük örnek bütçesiyle
Particle/Shape seçeneklerini ve zaman sürgüsünü deneyin. Sonuç beğenilirse aynı B dosyasını tekrar kullanabilirsiniz.
manifest.json istemi, seed'i, model revizyonlarını, ortamı, aşama sürelerini, sayıları ve koordinat dönüşümünü kaydeder.

Hata/kesinti: Drive bağlıyken **Tümünü çalıştır** yeniden denenebilir. Aynı klasör farklı ayarları kabul etmez.
Bozuk/yarım aşama yeniden üretilir; günlükler OUTPUT_DIR/logs ve /content/text-to-splat-v1/logs altındadır.


In [ ]:
result = Path(OUTPUT_DIR)
for name in ('target.ply', 'turntable.gif', 'input.png', 'cutout.png', 'manifest.json'):
    print('✓', result / name)
print((result / 'manifest.json').read_text(encoding='utf8'))